In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Github Repo Analysis (downstream of staging)
# MAGIC
# MAGIC This notebook reads from `github_api_staging` (populated by the
# MAGIC **Github Graph Traversal** notebook) and:
# MAGIC 1. Writes dependency files, contents, and commits to bronze Delta tables
# MAGIC 2. Computes aggregate stats per repo
# MAGIC 3. For repos marked `is_favorite`: AI-processes files + commits to generate insights
# MAGIC 4. Extracts related repositories from dependency manifests
# MAGIC
# MAGIC **Prerequisite**: Run the `Github Graph Traversal` notebook first to populate staging.

# COMMAND ----------

# DBTITLE 1,Configuration
dbutils.widgets.text("catalog", "bootcamp_students", "Unity Catalog name")
dbutils.widgets.text("schema", "abhibastia", "Schema name")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

# Source: CDC history table (mirrored from Lakebase via Change Data Feed)
repos_history_table = f"{catalog}.{schema}.lb_github_repos_history"

# Delta table targets
files_bronze_table = f"{catalog}.{schema}.github_repo_files_bronze"
commits_bronze_table = f"{catalog}.{schema}.github_repo_commits_bronze"
ai_insights_gold_table = f"{catalog}.{schema}.github_repo_ai_insights_gold"
file_contents_bronze_table = f"{catalog}.{schema}.github_file_contents_bronze"
related_repos_gold_table = f"{catalog}.{schema}.github_related_repos_gold"
github_staging_table = f"{catalog}.{schema}.github_api_staging"

print(f"Catalog: {catalog}")
print(f"Schema: {schema}")
print(f"Source (CDC): {repos_history_table}")
print(f"Files bronze: {files_bronze_table}")
print(f"File contents bronze: {file_contents_bronze_table}")
print(f"Commits bronze: {commits_bronze_table}")
print(f"AI insights gold: {ai_insights_gold_table}")
print(f"Related repos gold: {related_repos_gold_table}")
print(f"Staging table: {github_staging_table}")

# COMMAND ----------

# DBTITLE 1,Read current repos from CDC history table
import sys

# Read the latest state of each repo from the CDC history table.
# The history table contains all change events; we want the most recent
# non-deleted state per repo (latest _sort_by, excluding deletes).
repos_df = spark.sql(f"""
    WITH latest AS (
        SELECT *,
            ROW_NUMBER() OVER (PARTITION BY full_name ORDER BY _sort_by DESC) AS rn
        FROM {repos_history_table}
    )
    SELECT full_name, is_favorite, language, stargazers_count
    FROM latest
    WHERE rn = 1 AND _pg_change_type NOT IN('delete', 'update_preimage')
""")

repos = [row.asDict() for row in repos_df.collect()]

print(f"Found {len(repos)} current repos from {repos_history_table}")
for r in repos:
    fav = "⭐" if r["is_favorite"] else "  "
    print(f"  {fav} {r['full_name']} ({r['language']}, {r['stargazers_count']}★)")

# COMMAND ----------

# DBTITLE 1,Read from staging table (populated by Github Graph Traversal notebook)
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, BooleanType, TimestampType
)

# This notebook is purely downstream of the staging table.
# Run the "Github Graph Traversal" notebook first to populate it.
staging_df = spark.table(github_staging_table)

total_repos_staged = staging_df.filter("category = 'file'").select("repo_full_name").distinct().count()
file_count = staging_df.filter("category = 'file'").count()
content_count = staging_df.filter("category = 'content'").count()
commit_count = staging_df.filter("category = 'commit'").count()

print(f"✓ Staging table: {github_staging_table}")
print(f"✓ Total repos: {total_repos_staged}")
print(f"✓ Dependency file entries: {file_count}")
print(f"✓ Dependency file contents: {content_count}")
print(f"✓ Commits: {commit_count}")

# COMMAND ----------

# DBTITLE 1,Write files to bronze Delta table (from staging)
if file_count > 0:
    files_df = (
        staging_df.filter("category = 'file'")
        .select(
            "repo_full_name", "path",
            F.col("file_type").alias("type"),
            "sha", "size", "mode", "tree_truncated",
            F.to_timestamp("ingested_at").alias("ingested_at")
        )
    )

    (
        files_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(files_bronze_table)
    )
    print(f"✓ Wrote {file_count} file entries to {files_bronze_table}")
    display(files_df.groupBy("repo_full_name").count().orderBy("count", ascending=False))
else:
    print("No file data to write.")

# COMMAND ----------

# DBTITLE 1,Write file contents to bronze Delta table (from staging)
if content_count > 0:
    contents_df = (
        staging_df.filter("category = 'content'")
        .select(
            "repo_full_name", "path", "content", "encoding",
            "sha", "size", "language",
            F.to_timestamp("ingested_at").alias("ingested_at")
        )
    )

    (
        contents_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(file_contents_bronze_table)
    )
    print(f"✓ Wrote {content_count} file contents to {file_contents_bronze_table}")
    display(contents_df.groupBy("repo_full_name").count().orderBy("count", ascending=False))
else:
    print("No file content data to write.")

# COMMAND ----------

# DBTITLE 1,Write commits to bronze Delta table (from staging)
if commit_count > 0:
    commits_df = (
        staging_df.filter("category = 'commit'")
        .select(
            "repo_full_name", "sha",
            "author_name", "author_email",
            F.to_timestamp("author_date").alias("author_date"),
            "committer_name", "committer_email",
            F.to_timestamp("committer_date").alias("committer_date"),
            "message", "additions", "deletions",
            F.to_timestamp("ingested_at").alias("ingested_at")
        )
    )

    (
        commits_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(commits_bronze_table)
    )
    print(f"✓ Wrote {commit_count} commits to {commits_bronze_table}")
    display(commits_df.groupBy("repo_full_name").count().orderBy("count", ascending=False))
else:
    print("No commit data to write.")

# COMMAND ----------

# DBTITLE 1,Compute heavy stats per repo
from pyspark.sql.window import Window

# --- File stats ---
if file_count > 0:
    files_stats = (
        spark.table(files_bronze_table)
        .filter(F.col("type") == "blob")  # Only count actual files, not directories
        .groupBy("repo_full_name")
        .agg(
            F.count("*").alias("total_files"),
            F.sum("size").alias("total_size_bytes"),
            F.avg("size").alias("avg_file_size_bytes"),
            F.max("size").alias("max_file_size_bytes"),
            # Count by file extension
            F.countDistinct(
                F.regexp_extract("path", r"\.([^.]+)$", 1)
            ).alias("unique_extensions"),
        )
    )

# --- Commit stats ---
if commit_count > 0:
    commits_stats = (
        spark.table(commits_bronze_table)
        .groupBy("repo_full_name")
        .agg(
            F.count("*").alias("total_commits"),
            F.countDistinct("author_name").alias("unique_authors"),
            F.countDistinct("author_email").alias("unique_author_emails"),
            F.min("author_date").alias("first_commit_date"),
            F.max("author_date").alias("last_commit_date"),
            F.sum("additions").alias("total_additions"),
            F.sum("deletions").alias("total_deletions"),
            F.avg("additions").alias("avg_additions_per_commit"),
            F.avg("deletions").alias("avg_deletions_per_commit"),
            # Commit message length stats
            F.avg(F.length("message")).alias("avg_message_length"),
        )
    )

    # Days active = last commit - first commit
    commits_stats = commits_stats.withColumn(
        "days_active",
        F.datediff("last_commit_date", "first_commit_date")
    ).withColumn(
        "commits_per_day",
        F.when(F.col("days_active") > 0,
               F.round(F.col("total_commits") / F.col("days_active"), 2))
        .otherwise(F.col("total_commits"))
    )

# --- Join file + commit stats ---
if file_count > 0 and commit_count > 0:
    repo_stats = files_stats.join(commits_stats, on="repo_full_name", how="outer")
elif file_count > 0:
    repo_stats = files_stats
elif commit_count > 0:
    repo_stats = commits_stats
else:
    dbutils.notebook.exit("No data collected - nothing to process.")

repo_stats = repo_stats.withColumn("processed_at", F.current_timestamp())
display(repo_stats)

# COMMAND ----------

# MAGIC %md
# MAGIC ## AI Processing for Favorite Repos
# MAGIC
# MAGIC For repos marked `is_favorite`, we send the file tree structure and
# MAGIC commit history to a Foundation Model to generate:
# MAGIC - A purpose/description summary
# MAGIC - Key technology stack identification
# MAGIC - Commit pattern analysis
# MAGIC - A "health score" (0-100)

# COMMAND ----------

# DBTITLE 1,AI Processing - Favorites Only
import json

favorite_repos = [r for r in repos if r["is_favorite"]]
print(f"Found {len(favorite_repos)} favorite repos for AI processing")

if not favorite_repos:
    print("No favorite repos - skipping AI processing.")
    dbutils.notebook.exit("No favorites to AI-process. Stats written to bronze tables.")

# Collect data for favorites from staging (only what we need for AI prompts)
favorite_names = [r["full_name"] for r in favorite_repos]

all_files = [
    row.asDict() for row in
    staging_df.filter((F.col("category") == "file") & F.col("repo_full_name").isin(favorite_names))
    .select("repo_full_name", "path", F.col("file_type").alias("type"), "sha", "size", "mode", "tree_truncated", "ingested_at")
    .collect()
]
all_file_contents = [
    row.asDict() for row in
    staging_df.filter((F.col("category") == "content") & F.col("repo_full_name").isin(favorite_names))
    .select("repo_full_name", "path", "content", "encoding", "sha", "size", "language", "ingested_at")
    .collect()
]
all_commits = [
    row.asDict() for row in
    staging_df.filter((F.col("category") == "commit") & F.col("repo_full_name").isin(favorite_names))
    .select("repo_full_name", "sha", "author_name", "author_email", "author_date",
            "committer_name", "committer_email", "committer_date", "message",
            "additions", "deletions", "ingested_at")
    .collect()
]
print(f"Collected for AI: {len(all_files)} files, {len(all_file_contents)} contents, {len(all_commits)} commits")


def build_ai_prompt(full_name: str, files: list[dict], commits: list[dict], file_contents: list[dict]) -> str:
    """Build a structured prompt for the AI model from repo files, commits, and contents."""

    # Summarize file tree (top-level structure + extensions)
    file_paths = [f["path"] for f in files if f.get("type") == "blob"]
    top_level = sorted(set(p.split("/")[0] for p in file_paths))[:30]

    # Count extensions
    extensions = {}
    for p in file_paths:
        ext = p.rsplit(".", 1)[-1] if "." in p else "(none)"
        extensions[ext] = extensions.get(ext, 0) + 1
    top_extensions = sorted(extensions.items(), key=lambda x: -x[1])[:15]

    # Recent commit messages (last 30)
    recent_messages = [c.get("message", "")[:200] for c in commits[:30]]

    # Unique authors
    authors = list(set(c.get("author_name", "Unknown") for c in commits if c.get("author_name")))[:20]

    # Key file contents - prioritize config/entry files, limit total size
    key_files = []
    priority_patterns = [
        "README", "setup.py", "pyproject.toml", "package.json", "Cargo.toml",
        "go.mod", "build.gradle", "pom.xml", "Makefile", "Dockerfile",
        "requirements.txt", "setup.cfg", "app.py", "main.py", "index.ts",
        "index.js", "lib.rs", "mod.rs",
    ]

    # Sort contents: priority files first, then by path
    def file_priority(fc):
        name = fc["path"].rsplit("/", 1)[-1]
        for i, pat in enumerate(priority_patterns):
            if pat.lower() in name.lower():
                return i
        return len(priority_patterns)

    sorted_contents = sorted(file_contents, key=file_priority)

    # Include up to ~30KB of file content in the prompt
    content_budget = 30_000
    content_used = 0
    for fc in sorted_contents:
        content = fc.get("content", "")
        if not content:
            continue
        # Truncate individual files to 5KB
        truncated = content[:5000]
        if content_used + len(truncated) > content_budget:
            break
        key_files.append(f"### {fc['path']}\n```\n{truncated}\n```")
        content_used += len(truncated)

    file_contents_section = ""
    if key_files:
        file_contents_section = f"""

## Key File Contents ({len(key_files)} files)
{chr(10).join(key_files)}
"""

    prompt = f"""Analyze this GitHub repository: {full_name}

## File Structure
Total files: {len(file_paths)}
Top-level entries: {', '.join(top_level)}

File extensions (count):
{chr(10).join(f'  .{ext}: {count}' for ext, count in top_extensions)}
{file_contents_section}
## Commit History
Total commits (up to 200): {len(commits)}
Unique authors: {', '.join(authors[:10])}

Recent commit messages:
{chr(10).join(f'  - {msg.split(chr(10))[0]}' for msg in recent_messages)}

## Task
Based on this information (including the actual file contents), provide a JSON response with:
1. "purpose": A 2-3 sentence summary of what this repository does
2. "tech_stack": List of key technologies/frameworks/languages used
3. "commit_patterns": Brief analysis of commit activity and collaboration patterns
4. "strengths": 2-3 notable strengths of the project
5. "health_score": Integer 0-100 rating the project health (activity, maintenance, structure)
6. "health_rationale": One sentence explaining the health score
7. "code_quality_notes": 2-3 observations about code quality from the actual source

Respond ONLY with valid JSON, no markdown formatting."""

    return prompt


def call_foundation_model(prompt: str) -> dict:
    """Call a Foundation Model using Databricks ai_query()."""
    result_df = spark.sql("""
        SELECT ai_query(
            'databricks-claude-sonnet-4',
            CONCAT(
                'You are a senior software engineer analyzing GitHub repositories. '
                'Always respond with valid JSON only, no markdown formatting.\n\n',
                :prompt
            )
        ) AS response
    """, args={"prompt": prompt})

    content = result_df.collect()[0]["response"]

    # Parse the JSON response (strip markdown code fences if present)
    content = content.strip()
    if content.startswith("```"):
        content = content.split("\n", 1)[1]  # Remove first line
        content = content.rsplit("```", 1)[0]  # Remove last fence

    return json.loads(content)


# Process each favorite repo
ai_insights = []

for repo in favorite_repos:
    full_name = repo["full_name"]
    print(f"\n🤖 AI-processing: {full_name}")

    # Get files, commits, and contents for this repo
    repo_files = [f for f in all_files if f["repo_full_name"] == full_name]
    repo_commits = [c for c in all_commits if c["repo_full_name"] == full_name]
    repo_contents = [f for f in all_file_contents if f["repo_full_name"] == full_name]

    print(f"   Files: {len(repo_files)}, Commits: {len(repo_commits)}, Contents: {len(repo_contents)}")

    try:
        prompt = build_ai_prompt(full_name, repo_files, repo_commits, repo_contents)
        insights = call_foundation_model(prompt)

        ai_insights.append({
            "repo_full_name": full_name,
            "purpose": insights.get("purpose", ""),
            "tech_stack": json.dumps(insights.get("tech_stack", [])),
            "commit_patterns": insights.get("commit_patterns", ""),
            "strengths": json.dumps(insights.get("strengths", [])),
            "health_score": int(insights.get("health_score", 0)),
            "health_rationale": insights.get("health_rationale", ""),
            "code_quality_notes": json.dumps(insights.get("code_quality_notes", [])),
            "processed_at": datetime.utcnow().isoformat(),
        })
        print(f"   ✓ Health score: {insights.get('health_score')}/100")
        print(f"   ✓ Purpose: {insights.get('purpose', '')[:100]}...")

    except Exception as e:
        print(f"   ✗ AI processing failed: {e}")
        ai_insights.append({
            "repo_full_name": full_name,
            "purpose": f"AI processing failed: {str(e)[:200]}",
            "tech_stack": "[]",
            "commit_patterns": "",
            "strengths": "[]",
            "health_score": -1,
            "health_rationale": "Processing error",
            "code_quality_notes": "[]",
            "processed_at": datetime.utcnow().isoformat(),
        })

print(f"\n\nAI processing complete: {len(ai_insights)} repos processed")

# COMMAND ----------

# DBTITLE 1,Write AI insights to gold Delta table
from pyspark.sql.types import IntegerType

ai_insights_schema = StructType([
    StructField("repo_full_name", StringType(), False),
    StructField("purpose", StringType(), True),
    StructField("tech_stack", StringType(), True),
    StructField("commit_patterns", StringType(), True),
    StructField("strengths", StringType(), True),
    StructField("health_score", IntegerType(), True),
    StructField("health_rationale", StringType(), True),
    StructField("code_quality_notes", StringType(), True),
    StructField("processed_at", StringType(), True),
])

if ai_insights:
    insights_df = spark.createDataFrame(ai_insights, schema=ai_insights_schema)
    insights_df = insights_df.withColumn("processed_at", F.to_timestamp("processed_at"))

    (
        insights_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(ai_insights_gold_table)
    )
    print(f"✓ Wrote {insights_df.count()} AI insight rows to {ai_insights_gold_table}")
    display(insights_df)
else:
    print("No AI insights to write.")

# COMMAND ----------

# DBTITLE 1,Related Repositories (from dependency files)
# Extract related repositories by parsing dependency manifest files
# (package.json, requirements.txt, pyproject.toml, etc.) and using AI
# to map dependencies to their GitHub repositories.

DEPENDENCY_FILES = {
    "package.json", "requirements.txt", "setup.py", "pyproject.toml",
    "Cargo.toml", "go.mod", "build.gradle", "pom.xml", "Gemfile",
    "composer.json", "setup.cfg", "Pipfile",
}

related_repos = []

for repo in favorite_repos:
    full_name = repo["full_name"]
    repo_contents = [f for f in all_file_contents if f["repo_full_name"] == full_name]

    # Filter to only dependency/manifest files
    dep_files = [
        f for f in repo_contents
        if f.get("content")
        and f["path"].rsplit("/", 1)[-1] in DEPENDENCY_FILES
    ]

    if not dep_files:
        print(f"\n📦 {full_name}: No dependency files found, skipping.")
        continue

    print(f"\n📦 {full_name}: Found {len(dep_files)} dependency files: {[f['path'] for f in dep_files]}")

    # Combine all dependency file contents into one prompt
    dep_contents = "\n".join(
        f"### {f['path']}\n```\n{f['content'][:10000]}\n```"
        for f in dep_files
    )
    # Truncate total to fit token budget
    dep_contents = dep_contents[:30000]

    try:
        related_df = spark.sql("""
            SELECT ai_query(
                'databricks-claude-sonnet-4',
                CONCAT(
                    'You are analyzing dependency files from a GitHub repository to identify related repositories. '
                    'Extract all dependencies/packages listed and map them to their GitHub repositories. '
                    'Only include packages that have a known GitHub repository. '
                    'Respond ONLY with a JSON array of objects: '
                    '{"package_name": "the dependency name as listed", '
                    '"github_repo": "owner/repo format", '
                    '"source_file": "which dependency file listed it", '
                    '"relationship": "dependency|devDependency|build|peer"}. '
                    'If unsure of the GitHub repo for a package, omit it.\n\n'
                    'Repository: ', :full_name, '\n\n', :dep_contents
                )
            ) AS response
        """, args={"full_name": full_name, "dep_contents": dep_contents})

        raw_response = related_df.collect()[0]["response"].strip()
        if raw_response.startswith("```"):
            raw_response = raw_response.split("\n", 1)[1]
            raw_response = raw_response.rsplit("```", 1)[0]

        repos_found = json.loads(raw_response)

        for r in repos_found:
            related_repos.append({
                "repo_full_name": full_name,
                "package_name": r.get("package_name", ""),
                "related_github_repo": r.get("github_repo", ""),
                "source_file": r.get("source_file", ""),
                "relationship": r.get("relationship", "dependency"),
                "extracted_at": datetime.utcnow().isoformat(),
            })

        print(f"   Found {len(repos_found)} related repositories")

    except Exception as e:
        print(f"   ✗ Related repo extraction failed: {e}")

print(f"\n📦 Total related repos identified: {len(related_repos)}")

# Write related repositories to separate gold table
if related_repos:
    rel_schema = StructType([
        StructField("repo_full_name", StringType(), False),
        StructField("package_name", StringType(), True),
        StructField("related_github_repo", StringType(), True),
        StructField("source_file", StringType(), True),
        StructField("relationship", StringType(), True),
        StructField("extracted_at", StringType(), True),
    ])
    rel_df = spark.createDataFrame(related_repos, schema=rel_schema)
    rel_df = rel_df.withColumn("extracted_at", F.to_timestamp("extracted_at"))

    (
        rel_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(related_repos_gold_table)
    )
    print(f"✓ Wrote {rel_df.count()} related repo entries to {related_repos_gold_table}")
    display(rel_df.groupBy("repo_full_name", "relationship").count().orderBy("repo_full_name"))
else:
    print("No related repos to write.")

# COMMAND ----------

# DBTITLE 1,Final Summary
print("=" * 60)
print("DAY 2 PROCESSING COMPLETE")
print("=" * 60)
print(f"\n🗃️  Staging table:    {github_staging_table} ({total_repos_staged} repos)")
print(f"\n📁 Files bronze:      {files_bronze_table}")
print(f"   → {file_count} file entries across {total_repos_staged} repos")
print(f"\n📄 File contents:     {file_contents_bronze_table}")
print(f"   → {content_count} files with content fetched")
print(f"\n📝 Commits bronze:    {commits_bronze_table}")
print(f"   → {commit_count} commits across {total_repos_staged} repos")
print(f"\n🤖 AI insights gold:  {ai_insights_gold_table}")
print(f"   → {len(ai_insights)} favorite repos AI-analyzed")
print(f"\n📦 Related repos gold:  {related_repos_gold_table}")
print(f"   → {len(related_repos)} related repos identified from dependency files")
print(f"\n⭐ Favorites processed:")
for insight in ai_insights:
    score = insight["health_score"]
    emoji = "🟢" if score >= 70 else "🟡" if score >= 40 else "🔴" if score >= 0 else "❌"
    print(f"   {emoji} {insight['repo_full_name']}: {score}/100")
print("=" * 60)
